# 00_UnitCatalog_Bootstrap_&_Audit_log

**Purpose**

Initializes the Healthcare Lakehouse environment before any data pipelines execute.

**This notebook:
**
Creates the Unity Catalog schemas (Bronze, Silver, OMOP, Gold) if they do not already exist.
Creates the Gold pipeline_audit table.
Defines the reusable log_pipeline_run() function that is used by all pipeline notebooks for execution logging.

**Inputs**

None.

This notebook only initializes the Lakehouse metadata and reusable utilities.

**Outputs**
- Unity Catalog schemas:
    - bronze
    - silver
    - omop
    - gold
- Gold audit table:
    - db_healthcare_kl.gold.pipeline_audit
- Reusable Python function:
    - log_pipeline_run()

**Runtime Estimate**
~5–10 seconds

**Dependencies**
- Executed after 00_Install Libraries
- Must run before:
- Bronze Layer
- Silver Layer
- OMOP Mapping
- Gold Layer
- ML Training
- Model Inference

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS db_healthcare_kl.bronze;
CREATE SCHEMA IF NOT EXISTS db_healthcare_kl.silver;
CREATE SCHEMA IF NOT EXISTS db_healthcare_kl.omop;
CREATE SCHEMA IF NOT EXISTS db_healthcare_kl.gold;

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS db_healthcare_kl.gold.pipeline_audit (
    pipeline_name STRING,
    layer STRING,
    status STRING,
    records_processed BIGINT,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    execution_time_seconds DOUBLE,
    executed_by STRING
)
USING DELTA
""")

In [0]:
from pyspark.sql import Row
from datetime import datetime

def log_pipeline_run(
    pipeline_name,
    layer,
    status,
    records_processed,
    start_time,
    end_time,
    executed_by="Prateek Garg"
):

    execution_time = (end_time - start_time).total_seconds()

    audit_df = spark.createDataFrame([
        Row(
            pipeline_name=pipeline_name,
            layer=layer,
            status=status,
            records_processed=records_processed,
            start_time=start_time,
            end_time=end_time,
            execution_time_seconds=execution_time,
            executed_by=executed_by
        )
    ])

    audit_df.write.mode("append").saveAsTable(
        "db_healthcare_kl.gold.pipeline_audit"
    )

In [0]:
display(
    spark.table("db_healthcare_kl.gold.pipeline_audit")
)